In [1]:
from PIL import Image
import math
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from __future__ import print_function
import argparse

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.autograd import Variable
from torch.utils.data import DataLoader
from model import Net as DBPNLL
from data import get_eval_set
from functools import reduce
from skimage.transform import pyramid_reduce, pyramid_expand

import scipy.io as sio
import time
import cv2

In [20]:
original = np.load('tropomi.npy')
original = np.expand_dims(original, axis = 2)

## Generate SR arrays

In [21]:
bic_tropomi = pyramid_reduce(original, downscale=16, sigma=None, order=3, mode='constant', cval=0, channel_axis=2)
bic_tropomi = np.squeeze(bic_tropomi, axis = 2)

In [22]:
np.save('Input/Set5_LR_x16/bic_tropomi.npy', bic_tropomi)

In [23]:
model_w_16 = 'weights/MOD_tensorese-hivemindDBPNLL1channel_16_MSE.pth'
model_w_8 = 'weights/MOD_tensorese-hivemindDBPNLL1channel_8_MSE.pth'
model_w_4 = 'weights/MOD_tensorese-hivemindDBPNLL1channel_4_MSE.pth'
model_w_2 = 'weights/MOD_tensorese-hivemindDBPNLL1channel_2_MSE.pth'

gpus_list=range(1)
cuda = True
if cuda and not torch.cuda.is_available():
    raise Exception("No GPU found, please run without --cuda")

torch.manual_seed(123)
if cuda:
    torch.cuda.manual_seed(123)

print('===> Loading datasets')
test_set = get_eval_set(os.path.join('Input','Set5_LR_x16'), 16)
testing_data_loader = DataLoader(dataset=test_set, num_workers=1, batch_size=1, shuffle=False)

print('===> Building models')
model_16 = DBPNLL(num_channels=1, base_filter=64,  feat = 256, num_stages=10, scale_factor=16)
model_8 = DBPNLL(num_channels=1, base_filter=64,  feat = 256, num_stages=10, scale_factor=8)
model_4 = DBPNLL(num_channels=1, base_filter=64,  feat = 256, num_stages=10, scale_factor=4)
model_2 = DBPNLL(num_channels=1, base_filter=64,  feat = 256, num_stages=10, scale_factor=2)

if cuda:
    model_16 = torch.nn.DataParallel(model_16, device_ids=gpus_list)
    model_8 = torch.nn.DataParallel(model_8, device_ids=gpus_list)
    model_4 = torch.nn.DataParallel(model_4, device_ids=gpus_list)
    model_2 = torch.nn.DataParallel(model_2, device_ids=gpus_list)

model_16.load_state_dict(torch.load(model_w_16, map_location=lambda storage, loc: storage))
model_8.load_state_dict(torch.load(model_w_8, map_location=lambda storage, loc: storage))
model_4.load_state_dict(torch.load(model_w_4, map_location=lambda storage, loc: storage))
model_2.load_state_dict(torch.load(model_w_2, map_location=lambda storage, loc: storage))
print('Pre-trained SR model is loaded.')

if cuda:
    model_16 = model_16.cuda(gpus_list[0])
    model_8 = model_8.cuda(gpus_list[0])
    model_4 = model_4.cuda(gpus_list[0])
    model_2 = model_2.cuda(gpus_list[0])

def eval():
    model_16.eval()
    model_8.eval()
    model_4.eval()
    model_2.eval()
    for batch in testing_data_loader:
        with torch.no_grad():
            input, name = Variable(batch[0]), batch[1]
        if cuda:
            input = input.cuda(gpus_list[0])

        t0 = time.time()
        with torch.no_grad():
            prediction_16 = model_16(input)
        t1 = time.time()
        name_16 = name[0].replace('.npy', '_16.npy')
        print("===> Processing: %s || Path: 1-16 || Timer: %.4f sec." % (name_16, (t1 - t0)))
        
        t0 = time.time()
        with torch.no_grad():
            prediction_8 = model_8(input)
            prediction_8_2 = model_2(prediction_8)
        t1 = time.time()
        name_8_2 = name[0].replace('.npy', '_8_2.npy')
        print("===> Processing: %s || Path: 1-8-16 || Timer: %.4f sec." % (name_8_2, (t1 - t0)))
        
        
        t0 = time.time()
        with torch.no_grad():
            prediction_4 = model_4(input)
            prediction_4_4 = model_4(prediction_4)
        t1 = time.time()
        name_4_4 = name[0].replace('.npy', '_4_4.npy')
        print("===> Processing: %s || Path: 1-4-16 || Timer: %.4f sec." % (name_4_4, (t1 - t0)))
        
        t0 = time.time()
        with torch.no_grad():
            prediction_2 = model_2(input)
            prediction_2_4 = model_4(prediction_2)
            prediction_2_4_2 = model_2(prediction_2_4)
        t1 = time.time()
        name_2_4_2 = name[0].replace('.npy', '_2_4_2.npy')
        print("===> Processing: %s || Path: 1-2-8-16 || Timer: %.4f sec." % (name_2_4_2, (t1 - t0)))
        
        t0 = time.time()
        with torch.no_grad():
            prediction_4 = model_4(input)
            prediction_4_2 = model_2(prediction_4)
            prediction_4_2_2 = model_2(prediction_4_2)
        t1 = time.time()
        name_4_2_2 = name[0].replace('.npy', '_4_2_2.npy')
        print("===> Processing: %s || Path: 1-4-8-16 || Timer: %.4f sec." % (name_4_2_2, (t1 - t0)))
        
        t0 = time.time()
        with torch.no_grad():
            prediction_2 = model_2(input)
            prediction_2_2 = model_2(prediction_2)
            prediction_2_2_4 = model_4(prediction_2_2)
        t1 = time.time()
        name_2_2_4 = name[0].replace('.npy', '_2_2_4.npy')
        print("===> Processing: %s || Path: 1-2-4-16 || Timer: %.4f sec." % (name_2_2_4, (t1 - t0)))
        
        t0 = time.time()
        with torch.no_grad():
            prediction_2 = model_2(input)
            prediction_2_2 = model_2(prediction_2)
            prediction_2_2_2 = model_2(prediction_2_2)
            prediction_2_2_2_2 = model_2(prediction_2_2_2)
        t1 = time.time()
        name_2_2_2_2 = name[0].replace('.npy', '_2_2_2_2.npy')
        print("===> Processing: %s || Path: 1-2-4-8-16 || Timer: %.4f sec." % (name_2_2_2_2, (t1 - t0)))
        
        t0 = time.time()
        with torch.no_grad():
            prediction_2 = model_2(input)
            prediction_2_8 = model_8(prediction_2)
        t1 = time.time()
        name_2_8 = name[0].replace('.npy', '_2_8.npy')
        print("===> Processing: %s || Path: 1-2-16 || Timer: %.4f sec." % (name_2_8, (t1 - t0)))
        

        save_img(prediction_16.cpu().data, name_16)
        save_img(prediction_8_2.cpu().data, name_8_2)
        save_img(prediction_4_4.cpu().data, name_4_4)
        save_img(prediction_2_4_2.cpu().data, name_2_4_2)
        save_img(prediction_4_2_2.cpu().data, name_4_2_2)
        save_img(prediction_2_2_4.cpu().data, name_2_2_4)
        save_img(prediction_2_2_2_2.cpu().data, name_2_2_2_2)
        save_img(prediction_2_8.cpu().data, name_2_8)
        
        
def save_img(img, img_name):
    save_arr = img.numpy().squeeze(axis = 0).squeeze(axis = 0)
    # save img
    save_dir=os.path.join('Results/','Set5_LR_x16')
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
        
    save_fn = save_dir +'/'+ img_name
    np.save(save_fn, save_arr)
    #cv2.imwrite(save_fn, cv2.cvtColor(save_img*255, cv2.COLOR_BGR2RGB),  [cv2.IMWRITE_PNG_COMPRESSION, 0])
    
eval()

===> Loading datasets
===> Building models
Pre-trained SR model is loaded.
===> Processing: bic_tropomi_16.npy || Path: 1-16 || Timer: 0.0070 sec.
===> Processing: bic_tropomi_8_2.npy || Path: 1-8-16 || Timer: 0.2893 sec.
===> Processing: bic_tropomi_4_4.npy || Path: 1-4-16 || Timer: 0.0111 sec.
===> Processing: bic_tropomi_2_4_2.npy || Path: 1-2-8-16 || Timer: 0.3154 sec.
===> Processing: bic_tropomi_4_2_2.npy || Path: 1-4-8-16 || Timer: 0.2740 sec.
===> Processing: bic_tropomi_2_2_4.npy || Path: 1-2-4-16 || Timer: 0.0385 sec.
===> Processing: bic_tropomi_2_2_2_2.npy || Path: 1-2-4-8-16 || Timer: 0.2946 sec.
===> Processing: bic_tropomi_2_8.npy || Path: 1-2-16 || Timer: 0.0109 sec.


## Export numpy arrays into images

In [24]:
def save_image(data, cm, fn):
   
    sizes = np.shape(data)
    height = float(sizes[0])
    width = float(sizes[1])
     
    fig = plt.figure()
    fig.set_size_inches(width/height, 1, forward=False)
    ax = plt.Axes(fig, [0., 0., 1., 1.])
    ax.set_axis_off()
    fig.add_axes(ax)
 
    ax.imshow(data, cmap=cm, vmin = 0, vmax = 1)
    plt.savefig(fn, dpi = height) 
    plt.close()

In [25]:
cmap = plt.cm.jet
for filename in os.listdir('Results/Set5_LR_x16/'):
    if filename != '.ipynb_checkpoints':
        sr_array = np.load('Results/Set5_LR_x16/'+filename)
        sr_array = np.flip(sr_array, 0)
        save_name = filename.replace('.npy', '_one.png')
        save_image(sr_array, cmap, '../SR_results/'+save_name)

## Errors

In [ ]:
# Compute realtive error (averaged per pixel)
abs_error = abs(SR_bic - np.squeeze(original, axis = 2))
#abs_error = np.mean(abs_error, axis=2)

# Plot absolute error
fig, axs = plt.subplots(1, 2)
fig.set_size_inches(10,5)

axs[0].imshow(original, aspect="auto")

axs[1].invert_yaxis()

cmap = plt.cm.plasma
im = axs[1].pcolormesh(abs_error, alpha=1, cmap = cmap, vmin = 0, vmax = 0.3)
plt.colorbar(im, ax = axs[1])

